# LayoutLM v3

## 1. Install necessary packages

In [1]:
!pip install torch transformers pandas pyarrow numpy seqeval tqdm seqeval
# !pip install seqeval tqdm

Defaulting to user installation because normal site-packages is not writeable


In [7]:
!pip install pymupdf

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 MB 58.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [pymupdf]m6/7 [pymupdf]ime]


## 2. Import packages

In [2]:
import json
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from seqeval.scheme import IOB2
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, LayoutLMv3ImageProcessor


SEED = 42
MODEL_NAME = "microsoft/layoutlmv3-base"

PARQUET_PATH = Path("../data/raw/docile.parquet")
OUTPUT_DIR = Path("../data/interim/layoutlm_experiment")

FIELDS = (
    "invoice_number",
    "invoice_date",
    "due_date",
    "seller_name",
    "client_name",
)

LABELS = ["O"] + [
    f"{prefix}-{field}"
    for field in FIELDS
    for prefix in ("B", "I")
]

LABEL2ID = {label: index for index, label in enumerate(LABELS)}
ID2LABEL = dict(enumerate(LABELS))

IGNORE_INDEX = -100
MAX_LENGTH = 512
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 3e-5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

/home/ubuntu/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## 3. Split documents

In [3]:
df = pd.read_parquet(PARQUET_PATH)

# print("Documents:", len(df))
# print("Columns:", df.columns.tolist())

# assert df["document_id"].is_unique
# assert df.index.is_unique

In [4]:
shuffled = df.copy()
# random.Random(SEED).shuffle(shuffled)

train_df, remaining_df = train_test_split(
    df,
    test_size=0.2,
    shuffle=True,
    random_state=SEED,
)

val_df, test_df = train_test_split(
    remaining_df,
    test_size=0.5,
    shuffle=True,
    random_state=SEED,
)
splits = {
    'train': train_df,
    'val': val_df,
    'test': test_df
}

In [18]:
# class DocumentSplitter:
#     """Keep each template cluster in one split."""

#     def split(self, frame, seed=42):
#         groups = defaultdict(list)

#         for index, row in frame.iterrows():
#             json_data = json.loads(row['json_data'])
#             cluster = json_data["metadata"].get("cluster_id")

#             if cluster is None or pd.isna(cluster):
#                 key = ("document", row["document_id"])
#             else:
#                 key = ("cluster", cluster)

#             groups[key].append(index)

#         keys = list(groups)
#         random.Random(seed).shuffle(keys)

#         if len(keys) < 10:
#             raise ValueError("Need at least 10 template groups for this split")

#         train_end = int(len(keys) * 0.8)
#         val_end = int(len(keys) * 0.9)

#         subsets = {
#             "train": keys[:train_end],
#             "val": keys[train_end:val_end],
#             "test": keys[val_end:],
#         }

#         return {
#             name: frame.loc[
#                 [index for key in subset for index in groups[key]]
#             ].copy()
#             for name, subset in subsets.items()
#         }

In [16]:
# row = df.iloc[0]


In [19]:
# splits = DocumentSplitter().split(df, seed=SEED)
# # a
# for name, frame in splits.items():
#     print(name, len(frame))

train 4657
val 414
test 609


### Match OCR-words and annotation by geometry

In [21]:
class InvoicePreparer:
    """Build page records from OCR words and annotation boxes."""

    def __init__(self, fields=FIELDS, min_coverage=0.5):
        self.fields = fields
        self.min_coverage = min_coverage

    @staticmethod
    def validate_box(value):
        box = np.asarray(value, dtype=np.float64)

        if box.shape != (4,) or not np.isfinite(box).all():
            raise ValueError(f"Invalid box: {value}")

        x0, y0, x1, y1 = box

        if not (0 <= x0 < x1 <= 1 and 0 <= y0 < y1 <= 1):
            raise ValueError(f"Invalid normalized coordinates: {value}")

        return box

    @staticmethod
    def word_coverage(word_box, annotation_box):
        top_left = np.maximum(word_box[:2], annotation_box[:2])
        bottom_right = np.minimum(word_box[2:], annotation_box[2:])

        intersection = np.maximum(bottom_right - top_left, 0).prod()
        word_area = np.prod(word_box[2:] - word_box[:2])

        return float(intersection / word_area)

    def prepare_document(self, row):
        json_data = json.loads(row.json_data)
        metadata = json_data["metadata"]
        page_count = int(metadata["page_count"])
        annotations = []

        for field in self.fields:
            annotation = json_data["invoice"].get(field)

            if annotation is None:
                continue

            if len(annotation["pages"]) != len(annotation["bbox"]):
                raise ValueError(f"Annotation lengths differ: {field}")

            for page, box in zip(annotation["pages"], annotation["bbox"]):
                page = int(page)

                if not 0 <= page < page_count:
                    raise ValueError(f"Invalid annotation page: {page}")

                annotations.append({
                    "field": field,
                    "page": page,
                    "box": self.validate_box(box),
                })

        words_by_page = defaultdict(list)
        matched_annotations = set()

        # Preserve the OCR order used by the parquet text offsets.
        words = sorted(row.text_with_boxes, key=lambda word: word["start"])

        for word in words:
            if not word["text"].strip():
                continue

            page = int(word["page"])

            if not 0 <= page < page_count:
                raise ValueError(f"Invalid OCR page: {page}")

            box = self.validate_box(word["bbox"])
            candidates = []

            for annotation_id, annotation in enumerate(annotations):
                if annotation["page"] != page:
                    continue

                coverage = self.word_coverage(box, annotation["box"])

                if coverage >= self.min_coverage:
                    candidates.append(annotation_id)

            if len(candidates) > 1:
                raise ValueError(
                    f"Ambiguous word: {word['text']!r}, page={page}"
                )

            owner = candidates[0] if candidates else None

            if owner is not None:
                matched_annotations.add(owner)

            words_by_page[page].append({
                "text": word["text"],
                "box": (box * 1000).astype(np.int64).tolist(),
                "owner": owner,
            })

        missing = [
            annotation["field"]
            for index, annotation in enumerate(annotations)
            if index not in matched_annotations
        ]

        if missing:
            raise ValueError(f"Annotations without OCR matches: {missing}")

        if not words_by_page:
            raise ValueError("Document has no OCR words")

        records = []

        for page, page_words in sorted(words_by_page.items()):
            labels = []
            previous_owner = None

            for word in page_words:
                owner = word["owner"]

                if owner is None:
                    label = "O"
                else:
                    prefix = "I" if owner == previous_owner else "B"
                    field = annotations[owner]["field"]
                    label = f"{prefix}-{field}"

                labels.append(LABEL2ID[label])
                previous_owner = owner

            records.append({
                "document_id": row.document_id,
                "pdf_path": str(row.pdf_path),
                "page": page,
                "words": [word["text"] for word in page_words],
                "boxes": [word["box"] for word in page_words],
                "labels": labels,
            })

        return records

    def build(self, frame):
        records = []
        rejected = []

        for row in tqdm(
            frame.itertuples(index=False),
            total=len(frame),
            desc="Prepare documents",
        ):
            try:
                document_records = self.prepare_document(row)
            except ValueError as error:
                rejected.append({
                    "document_id": row.document_id,
                    "reason": str(error),
                })
                continue

            records.extend(document_records)

        return records, pd.DataFrame(
            rejected,
            columns=["document_id", "reason"],
        )

In [22]:
preparer = InvoicePreparer()
pages_by_split = {}
rejected_by_split = {}

for name, frame in splits.items():
    pages, rejected = preparer.build(frame)

    pages_by_split[name] = pages
    rejected_by_split[name] = rejected

    accepted_documents = len({page["document_id"] for page in pages})

    print(
        name,
        "accepted documents:", accepted_documents,
        "pages:", len(pages),
        "rejected documents:", len(rejected),
    )

    if not pages:
        raise ValueError(f"No accepted pages in {name}")

Prepare documents:   0%|          | 6/4544 [00:00<01:18, 57.86it/s]

Prepare documents: 100%|██████████| 4544/4544 [01:25<00:00, 53.33it/s]


train accepted documents: 4380 pages: 5655 rejected documents: 164


Prepare documents: 100%|██████████| 568/568 [00:10<00:00, 51.79it/s]


val accepted documents: 552 pages: 716 rejected documents: 16


Prepare documents: 100%|██████████| 568/568 [00:11<00:00, 51.19it/s]

test accepted documents: 547 pages: 714 rejected documents: 21


In [23]:
rejected_by_split["train"].head(20)

,document_id,reason
0,8006950cfd1b473fbf27a0bc,"Ambiguous word: 'August', page=0"
1,d12a661e877d4ef4b1c7f979,Annotations without OCR matches: ['seller_name']
2,bae9600370a44f6d954bccaf,Annotations without OCR matches: ['invoice_date']
3,98e9cfc37c1a4e96949a19ee,Annotations without OCR matches: ['invoice_dat...
4,2c93ee1b82134ddd88fb28b7,Annotations without OCR matches: ['due_date']
5,7d47467707f2441a9ae7d2de,Annotations without OCR matches: ['invoice_date']
6,2034da2080554b7885ee01e7,"Ambiguous word: 'WNIS-AM', page=0"
7,2269a9fb4fd74cb69dc14216,"Ambiguous word: 'Media', page=0"
8,b89b759330574655aeeaca39,Annotations without OCR matches: ['invoice_date']
9,8ec2e70573b147238bdfe5c5,"Ambiguous word: 'July', page=0"


In [25]:
example = pages_by_split["train"][0]

display(pd.DataFrame({
    **example
}))

,document_id,pdf_path,page,words,boxes,labels
0,2e95329f4c9e48fca33538aa,/home/ubuntu/invoices-ocr/data/raw/docile/pdfs...,0,Aug,"[184, 92, 210, 102]",3
1,2e95329f4c9e48fca33538aa,/home/ubuntu/invoices-ocr/data/raw/docile/pdfs...,0,"20,","[215, 92, 232, 101]",4
2,2e95329f4c9e48fca33538aa,/home/ubuntu/invoices-ocr/data/raw/docile/pdfs...,0,20,"[238, 92, 252, 100]",4
3,2e95329f4c9e48fca33538aa,/home/ubuntu/invoices-ocr/data/raw/docile/pdfs...,0,CONT#,"[84, 106, 130, 114]",0
4,2e95329f4c9e48fca33538aa,/home/ubuntu/invoices-ocr/data/raw/docile/pdfs...,0,34264102,"[184, 106, 245, 114]",1
...,...,...,...,...,...,...
238,2e95329f4c9e48fca33538aa,/home/ubuntu/invoices-ocr/data/raw/docile/pdfs...,0,0.00,"[896, 855, 924, 863]",0
239,2e95329f4c9e48fca33538aa,/home/ubuntu/invoices-ocr/data/raw/docile/pdfs...,0,NSL,"[84, 870, 109, 878]",0
240,2e95329f4c9e48fca33538aa,/home/ubuntu/invoices-ocr/data/raw/docile/pdfs...,0,0.00,"[896, 870, 924, 879]",0
241,2e95329f4c9e48fca33538aa,/home/ubuntu/invoices-ocr/data/raw/docile/pdfs...,0,TOTAL,"[83, 885, 127, 894]",0


## 4. PyTorch Dataset

In [26]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)
image_processor = LayoutLMv3ImageProcessor.from_pretrained(
    MODEL_NAME,
    apply_ocr=False,
)

assert tokenizer.is_fast

In [27]:
df.columns

Index(['document_id', 'ocr_text', 'text_with_boxes', 'json_data', 'pdf_path',
       'split'],
      dtype='str')

In [28]:
import pymupdf
from PIL import Image

IMAGE_CACHE = Path("../data/interim/docile_images")


def load_page_image(page, dpi=150):
    cache_path = (
        IMAGE_CACHE
        / f"dpi_{dpi}"
        / page["document_id"]
        / f"page_{int(page['page']):04d}.png"
    )

    if cache_path.is_file():
        with Image.open(cache_path) as image:
            return image.convert("RGB")

    with pymupdf.open(page["pdf_path"]) as document:
        pdf_page = document.load_page(int(page["page"]))
        pixmap = pdf_page.get_pixmap(
            dpi=dpi,
            colorspace=pymupdf.csRGB,
            alpha=False,
        )
        image = Image.frombytes(
            "RGB",
            (pixmap.width, pixmap.height),
            pixmap.samples,
        )

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    image.save(cache_path)
    return image

In [29]:
pages_by_split['train'][0].keys()

dict_keys(['document_id', 'pdf_path', 'page', 'words', 'boxes', 'labels'])

In [31]:
for split, pages in pages_by_split.items():
    for page in tqdm(pages, desc=f"Cache {split} images"):
        load_page_image(page)

Cache test images: 100%|██████████| 714/714 [02:08<00:00,  5.57it/s]


In [32]:
class InvoiceDataset(Dataset):
    """Cache complete-word windows and retain page coordinates for evaluation."""

    def __init__(self, pages, tokenizer, image_processor, max_length=512):
        if not 4 <= max_length <= 512:
            raise ValueError("max_length must be between 4 and 512")

        self.pages = pages
        self.samples = []

        capacity = (
            max_length
            - tokenizer.num_special_tokens_to_add(pair=False)
        )

        for page_index, page in enumerate(tqdm(pages, desc="Tokenize pages")):
            encoded = tokenizer(
                page["words"],
                boxes=page["boxes"],
                add_special_tokens=False,
                truncation=False,
                return_attention_mask=False,
                return_token_type_ids=False,
            )
            image = load_page_image(page).convert("RGB")

            pixel_values = image_processor(
                images=image,
                return_tensors="pt",
            )["pixel_values"][0]

            pieces = [[] for _ in page["words"]]

            for token_id, word_id in zip(
                encoded["input_ids"],
                encoded.word_ids(),
            ):
                pieces[word_id].append(token_id)

            if any(not tokens or len(tokens) > capacity for tokens in pieces):
                raise ValueError(
                    f"Empty or overlong word: "
                    f"{page['document_id']}, page={page['page']}"
                )

            start = 0

            while start < len(pieces):
                end = start
                token_count = 0

                while (
                    end < len(pieces)
                    and token_count + len(pieces[end]) <= capacity
                ):
                    token_count += len(pieces[end])
                    end += 1

                input_ids = [tokenizer.cls_token_id]
                boxes = [[0, 0, 0, 0]]
                labels = [IGNORE_INDEX]
                word_ids = [-1]

                for word_id in range(start, end):
                    tokens = pieces[word_id]
                    continuation_count = len(tokens) - 1

                    input_ids.extend(tokens)
                    boxes.extend([page["boxes"][word_id]] * len(tokens))
                    labels.extend(
                        [page["labels"][word_id]]
                        + [IGNORE_INDEX] * continuation_count
                    )
                    word_ids.extend(
                        [word_id] + [-1] * continuation_count
                    )

                input_ids.append(tokenizer.sep_token_id)
                boxes.append([1000, 1000, 1000, 1000])
                labels.append(IGNORE_INDEX)
                word_ids.append(-1)

                attention_mask = [1] * len(input_ids)
                padding = max_length - len(input_ids)

                sample = {
                    "input_ids": (
                        input_ids + [tokenizer.pad_token_id] * padding
                    ),
                    "attention_mask": attention_mask + [0] * padding,
                    "bbox": boxes + [[0, 0, 0, 0]] * padding,
                    "labels": labels + [IGNORE_INDEX] * padding,
                    "word_ids": word_ids + [-1] * padding,
                    "page_index": page_index
                }
                sample = {
                    key: torch.tensor(value, dtype=torch.long)
                    for key, value in sample.items()
                }
                sample["pixel_values"] = pixel_values

                self.samples.append(sample)

                start = end

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]

In [33]:
datasets = {
    name: InvoiceDataset(pages, tokenizer, image_processor, MAX_LENGTH)
    for name, pages in pages_by_split.items()
}

loaders = {
    name: DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=name == "train",
        num_workers=0,
        pin_memory=device.type == "cuda",
    )
    for name, dataset in datasets.items()
}

batch = next(iter(loaders["train"]))

for name, tensor in batch.items():
    print(name, tuple(tensor.shape))

Tokenize pages: 100%|██████████| 714/714 [00:40<00:00, 17.64it/s]


input_ids (16, 512)
attention_mask (16, 512)
bbox (16, 512, 4)
labels (16, 512)
word_ids (16, 512)
page_index (16,)
pixel_values (16, 3, 224, 224)


## 5. Model wrapper

In [41]:
class LayoutLMFieldExtractor(nn.Module):
    """Predict one BIO label per text token using text, boxes, and page images."""

    def __init__(self, model_name, num_labels, id2label, label2id):
        super().__init__()

        self.backbone = AutoModel.from_pretrained(
            model_name,
            num_labels=num_labels,
            id2label=id2label,
            label2id=label2id,
        )
        self.dropout = nn.Dropout(
            self.backbone.config.hidden_dropout_prob
        )
        self.classifier = nn.Linear(
            self.backbone.config.hidden_size,
            num_labels,
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        bbox,
        pixel_values,
    ):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            bbox=bbox,
            pixel_values=pixel_values,
            return_dict=True,
        )

        # Visual tokens follow text tokens in the backbone output.
        text_hidden = outputs.last_hidden_state[:, :input_ids.shape[1]]

        return self.classifier(self.dropout(text_hidden))

In [42]:
model = LayoutLMFieldExtractor(
    model_name=MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(device)

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 5025.76it/s]


## 6. Metrics, `train_one_epoch`, `eval_one_epoch`, `train`

### Metrics

* `loss`
* `bio_word_accuracy`
* `field_word_micro_precision/recall/f1`
* `field_word_macro_f1`
* `per_field`
* `entity_precision/recall/f1`

Main metrcic for checkpoint selection `entity_f1`

In [36]:
class EpochMetrics:
    """Accumulate word metrics without retaining model outputs."""

    def __init__(self):
        self.confusion = torch.zeros(
            len(LABELS),
            len(LABELS),
            dtype=torch.long,
        )
        self.loss_sum = 0.0
        self.word_count = 0

    def update(self, logits, labels, loss):
        valid = labels != IGNORE_INDEX

        truth = labels[valid].detach().cpu()
        predicted = logits.argmax(dim=-1)[valid].detach().cpu()

        size = len(LABELS)
        counts = torch.bincount(
            truth * size + predicted,
            minlength=size * size,
        )

        self.confusion += counts.reshape(size, size)
        self.loss_sum += loss.detach().item() * truth.numel()
        self.word_count += truth.numel()

    def compute(self):
        if self.word_count == 0:
            raise ValueError("No supervised words")

        # Map O -> 0, B/I of each field -> one field ID.
        mapping = [0] + [
            field_id
            for field_id in range(1, len(FIELDS) + 1)
            for _ in range(2)
        ]

        matrix = torch.zeros(
            len(FIELDS) + 1,
            len(FIELDS) + 1,
            dtype=torch.float64,
        )

        for truth_id, truth_field in enumerate(mapping):
            for pred_id, pred_field in enumerate(mapping):
                matrix[truth_field, pred_field] += (
                    self.confusion[truth_id, pred_id]
                )

        tp = matrix.diag()[1:]
        fp = matrix.sum(dim=0)[1:] - tp
        fn = matrix.sum(dim=1)[1:] - tp

        precision = tp / (tp + fp).clamp_min(1)
        recall = tp / (tp + fn).clamp_min(1)
        f1 = 2 * tp / (2 * tp + fp + fn).clamp_min(1)

        active = (tp + fp + fn) > 0

        return {
            "loss": self.loss_sum / self.word_count,
            "bio_word_accuracy": (
                self.confusion.diag().sum().item() / self.word_count
            ),
            "field_word_micro_precision": (
                tp.sum() / (tp + fp).sum().clamp_min(1)
            ).item(),
            "field_word_micro_recall": (
                tp.sum() / (tp + fn).sum().clamp_min(1)
            ).item(),
            "field_word_micro_f1": (
                2 * tp.sum() / (2 * tp + fp + fn).sum().clamp_min(1)
            ).item(),
            "field_word_macro_f1": (
                f1[active].mean().item() if active.any() else 0.0
            ),
            "per_field": {
                field: {
                    "precision": precision[index].item(),
                    "recall": recall[index].item(),
                    "f1": f1[index].item(),
                    "support": int((tp + fn)[index].item()),
                }
                for index, field in enumerate(FIELDS)
            },
        }

In [37]:
def move_inputs(batch, device):
    return {
        key: batch[key].to(device, non_blocking=True)
        for key in ("input_ids", "attention_mask", "bbox", "pixel_values")
    }


def train_one_epoch(model, loader, optimizer, criterion, scheduler, device):
    model.train()
    metrics = EpochMetrics()

    for batch in tqdm(loader, desc="Train", leave=False):
        inputs = move_inputs(batch, device)
        labels = batch["labels"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(**inputs)
        loss = criterion(
            logits.reshape(-1, logits.shape[-1]),
            labels.reshape(-1),
        )

        if not torch.isfinite(loss):
            raise FloatingPointError("Training loss is not finite")

        loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
            error_if_nonfinite=True,
        )

        optimizer.step()
        scheduler.step()
        metrics.update(logits, labels, loss)

    return metrics.compute()


@torch.inference_mode()
def eval_one_epoch(model, loader, criterion, device):
    model.eval()

    metrics = EpochMetrics()
    word_predictions = {}

    for batch in tqdm(loader, desc="Evaluate", leave=False):
        inputs = move_inputs(batch, device)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model(**inputs)
        loss = criterion(
            logits.reshape(-1, logits.shape[-1]),
            labels.reshape(-1),
        )

        if not torch.isfinite(loss):
            raise FloatingPointError("Evaluation loss is not finite")

        metrics.update(logits, labels, loss)
        predicted = logits.argmax(dim=-1).cpu()

        for batch_index, page_index in enumerate(
            batch["page_index"].tolist()
        ):
            for token_index, word_index in enumerate(
                batch["word_ids"][batch_index].tolist()
            ):
                if word_index < 0:
                    continue

                key = (page_index, word_index)

                if key in word_predictions:
                    raise ValueError("A word was evaluated more than once")

                word_predictions[key] = predicted[
                    batch_index, token_index
                ].item()

    # Restore complete pages before measuring entity boundaries.
    true_sequences = []
    pred_sequences = []

    for page_index, page in enumerate(loader.dataset.pages):
        true_sequences.append([
            ID2LABEL[label]
            for label in page["labels"]
        ])

        pred_sequences.append([
            ID2LABEL[word_predictions[(page_index, word_index)]]
            for word_index in range(len(page["words"]))
        ])

    result = metrics.compute()

    for name, metric in (
        ("precision", precision_score),
        ("recall", recall_score),
        ("f1", f1_score),
    ):
        result[f"entity_{name}"] = metric(
            true_sequences,
            pred_sequences,
            mode="strict",
            scheme=IOB2,
            zero_division=0,
        )

    result["entity_report"] = classification_report(
        true_sequences,
        pred_sequences,
        mode="strict",
        scheme=IOB2,
        output_dict=True,
        zero_division=0,
    )

    return result

strict `IOB2` учитывает ошибки границ: например, `I-seller_name` после O не исправляется автоматически в `B-seller_name`.

In [38]:
def train(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterion,
    device,
    output_dir,
    epochs=5,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    model.to(device)

    history = []
    best_f1 = -1.0
    checkpoint_path = output_dir / "best.pt"

    for epoch in range(1, epochs + 1):
        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader, 
            optimizer=optimizer,
            criterion=criterion,
            scheduler=scheduler,
            device=device
        )
        val_metrics = eval_one_epoch(
            model=model,
            loader=val_loader, 
            criterion=criterion,
            device=device
        )
        epoch_metrics = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            **{
                f"validation_{key}": value
                for key, value in val_metrics.items()
                if key != "per_entity"
            },
        }

        history.append(epoch_metrics)
        pd.DataFrame(history).to_csv(
            output_dir / "history.csv",
            index=False,
        )

        print(
            f"Epoch {epoch}/{epochs} | "
            f"train loss={train_metrics['loss']:.4f} | "
            f"val loss={val_metrics['loss']:.4f} | "
            f"val entity F1={val_metrics['entity_f1']:.4f}"
        )

        if val_metrics["entity_f1"] > best_f1:
            best_f1 = float(val_metrics["entity_f1"])

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "model_name": MODEL_NAME,
                    "labels": LABELS,
                    "fields": list(FIELDS),
                    "max_length": MAX_LENGTH,
                    "epoch": epoch,
                    "val_entity_f1": best_f1,
                },
                checkpoint_path,
            )

        # (output_dir / "history.json").write_text(
        #     json.dumps(history, indent=2),
        #     encoding="utf-8",
        # )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True,
    )

    model.load_state_dict(checkpoint["model_state_dict"])

    return pd.DataFrame(history)

In [52]:
# import importlib.metadata


# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# tokenizer.save_pretrained(OUTPUT_DIR / "tokenizer")

# config = {
#     "model_name": MODEL_NAME,
#     "fields": list(FIELDS),
#     "labels": LABELS,
#     "seed": SEED,
#     "max_length": MAX_LENGTH,
#     "batch_size": BATCH_SIZE,
#     "epochs": EPOCHS,
#     "learning_rate": LEARNING_RATE,
#     "min_word_coverage": preparer.min_coverage,
#     "split": "template_cluster_80_10_10",
#     "versions": {
#         package: importlib.metadata.version(package)
#         for package in (
#             "torch", "transformers", "pandas", "numpy", "seqeval"
#         )
#     },
# }

# (OUTPUT_DIR / "config.json").write_text(
#     json.dumps(config, indent=2),
#     encoding="utf-8",
# )

713

In [53]:
# split_manifest = []

# for name, frame in splits.items():
#     rejected_ids = set(rejected_by_split[name]["document_id"])

#     split_manifest.extend({
#         "document_id": document_id,
#         "split": name,
#         "accepted": document_id not in rejected_ids,
#     } for document_id in frame["document_id"])

#     rejected_by_split[name].to_csv(
#         OUTPUT_DIR / f"{name}_rejected.csv",
#         index=False,
#     )

# pd.DataFrame(split_manifest).to_csv(
#     OUTPUT_DIR / "split_manifest.csv",
#     index=False,
# )

In [43]:
from transformers import get_linear_schedule_with_warmup

epochs = 20
learning_rate = 3e-5
decay = []
no_decay = []

model = LayoutLMFieldExtractor(
    model_name=MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(device)

for name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue

    if parameter.ndim < 2:
        no_decay.append(parameter)
    else:
        decay.append(parameter)

optimizer = torch.optim.AdamW(
    [
        {"params": decay, "weight_decay": 0.01},
        {"params": no_decay, "weight_decay": 0.0},
    ],
    lr=learning_rate,
)

total_steps = epochs * len(loaders["train"])
warmup_steps = int(0.1 * total_steps)  # Start with 10% warmup.

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 5448.58it/s]


In [ ]:
history = train(
    model=model,
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,

    output_dir=OUTPUT_DIR,
    epochs=epochs
)

Epoch 1/20 | train loss=0.3576 | val loss=0.0408 | val entity F1=0.7647


Epoch 2/20 | train loss=0.0312 | val loss=0.0271 | val entity F1=0.8584


Epoch 3/20 | train loss=0.0221 | val loss=0.0263 | val entity F1=0.8685


Epoch 4/20 | train loss=0.0176 | val loss=0.0223 | val entity F1=0.8963


Epoch 5/20 | train loss=0.0142 | val loss=0.0240 | val entity F1=0.9018


Epoch 6/20 | train loss=0.0120 | val loss=0.0238 | val entity F1=0.8952


Train:  43%|████▎     | 220/511 [02:56<03:52,  1.25it/s]

In [55]:
history.to_csv('../data/interim/history_20_epochs.csv')

NameError: name 'history' is not defined

In [ ]:
test_metrics = eval_one_epoch(
    model=model,
    loader=loaders["test"],
    device=device,
)

display(pd.Series({
    key: value
    for key, value in test_metrics.items()
    if not isinstance(value, dict)
}))

display(pd.DataFrame(test_metrics["per_field"]).T)
display(pd.DataFrame(test_metrics["entity_report"]).T)

(OUTPUT_DIR / "test_metrics.json").write_text(
    json.dumps(test_metrics, indent=2),
    encoding="utf-8",
)